In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\pramo\AppData\Local\Temp\ipykernel_3604\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [11]:
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,doc_content_chars_max=200
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [12]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
web_doc = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 250
)
chunks = text_splitter.split_documents(web_doc)
# embeddings = OllamaEmbeddings(model="nano")
vector_store = FAISS.from_documents(chunks, OllamaEmbeddings(model="nomic-embed-text"))
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023F739BEAD0>, search_kwargs={'k': 3})

In [13]:
from langchain_core.tools import create_retriever_tool

langsmith_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "search for information about langsmith. For any question about langsmith, you must use this tool"
)

In [14]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_search_max = 200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)


In [15]:
tools=[
    wiki_tool,
    langsmith_tool,
    arxiv_tool
]

In [17]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)


In [25]:
##pulling promts from langchain hub
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent",
    dangerously_pull_public_prompt=True
)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [ ]:
##creating agents 
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    tools = tools,
    prompt=prompt
)
